# 承認済み部位別 before / after を解析する

`video_region_pair_candidates.ipynb` で固定した `selected_region_pairs.json` を使います。
現在選択されている部位だけを解析し、別部位のペアへ自動で置き換えません。


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_selected_regions.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('repo root:', Path.cwd())


In [ ]:
VIDEO = Path('makeup0923.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SELECTED = OUTPUT / 'selected_region_pairs.json'
ANALYSIS_ROOT = OUTPUT / 'selected_region_analysis'

if not SELECTED.is_file():
    raise FileNotFoundError(f'部位別ペアが固定されていません: {SELECTED}')
print('selected:', SELECTED)


In [ ]:
from analysis.analyze_selected_regions import analyze_selected_regions
from IPython.display import HTML, display
import base64

summary = analyze_selected_regions(SELECTED, ANALYSIS_ROOT)
RUN_DIR = Path(summary['output_dir'])
print('analysis:', RUN_DIR)

# report.html 内の相対画像パスは VS Code/Jupyter の HTML 表示では
# notebook 側を基準に解決されて壊れるため、表示時だけ data URI に埋め込む。
report_html = (RUN_DIR / 'report.html').read_text(encoding='utf-8')
for region in summary['regions']:
    for phase in ('before', 'after'):
        filename = f'{region}_{phase}.png'
        image_path = RUN_DIR / filename
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        encoded = base64.b64encode(image_path.read_bytes()).decode('ascii')
        report_html = report_html.replace(
            f"src='{filename}'",
            f"src='data:image/png;base64,{encoded}'",
        )
display(HTML(report_html))


In [ ]:
for region, result in summary['regions'].items():
    print('\n', region)
    for row in result['deltas']:
        before = row['before']
        after = row['after']
        delta = row['delta']
        print(f"{row['label']}: {before:.4f} -> {after:.4f}  delta={delta:+.4f}")
print('\nreport:', RUN_DIR / 'report.html')
print('csv   :', RUN_DIR / 'feature_deltas.csv')


## 目の下 ROI を拡大して確認する

左右それぞれの目の下 ROI を before / after で拡大表示します。黄色が実際の計測領域です。


In [ ]:
from pathlib import Path
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# notebook を repo 直下 / notebooks/ のどちらから実行しても同じ場所を指す。
cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_selected_regions.py')
if (cwd / marker).is_file():
    project_root = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    project_root = cwd.parent
    os.chdir(project_root)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.analyze_selected_regions import analyze_selected_regions, _load_phase

video = Path('makeup0923.mp4')
output = Path('outputs') / video.stem
selected_path = output / 'selected_region_pairs.json'
analysis_root = output / 'selected_region_analysis'
if not selected_path.is_file():
    raise FileNotFoundError(selected_path)

closeup_summary = analyze_selected_regions(selected_path, analysis_root)
region = 'eye_texture'
if region not in closeup_summary['regions']:
    raise ValueError(f'{region} が selected_region_pairs.json にありません。')
selection = closeup_summary['regions'][region]['selection']
before_img, before_masks, _ = _load_phase(selection['before'], region)
after_img, after_masks, _ = _load_phase(selection['after'], region)

def make_closeup(image, mask, pad_face_fraction=0.035):
    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('ROI mask is empty')
    face_width = xs.max() - xs.min() + 1
    pad = max(12, round(face_width * pad_face_fraction))
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(image.shape[0], int(ys.max()) + pad + 1)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(image.shape[1], int(xs.max()) + pad + 1)
    crop = image[y0:y1, x0:x1].copy()
    local_mask = mask[y0:y1, x0:x1]
    fill = crop.copy()
    fill[local_mask] = (50, 220, 240)
    overlay = cv2.addWeighted(fill, 0.35, crop, 0.65, 0)
    contours = cv2.findContours(local_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    cv2.drawContours(overlay, contours, -1, (0, 200, 255), 1, cv2.LINE_AA)
    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

mask_names = [
    ('screen_left_lower_eye_skin', '画面左目の下'),
    ('screen_right_lower_eye_skin', '画面右目の下'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
for row, (mask_name, label) in enumerate(mask_names):
    axes[row, 0].imshow(make_closeup(before_img, before_masks[mask_name]))
    axes[row, 0].set_title(f'before / {label}')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(make_closeup(after_img, after_masks[mask_name]))
    axes[row, 1].set_title(f'after / {label}')
    axes[row, 1].axis('off')

run_dir = Path(closeup_summary['output_dir'])
closeup_path = run_dir / 'eye_texture_closeup.png'
fig.savefig(closeup_path, dpi=180, bbox_inches='tight')
plt.show()
print('closeup:', closeup_path)
